# JEPX 전력 가격 예측 — LightGBM 모델 A (2026-11-17)

**목표**: 도쿄의 30분 단위 다음 날 전력 가격 `y_price`를 예측하고, 팀의 9~10월 검증 베이스라인(MAE 1.832363)과 비교한다.

- **학습**: 2024-04-01 ~ 2025-08-31
- **검증**: 2025-09-01 ~ 2025-10-31
- **테스트**: 2025-11-01 ~ 2026-10-31 (**예측값만 생성, test MAE 평가/튜닝 금지**)
- **모델 A**: OCCTO 관련 변수 제외. `date`, `slot`, `slot_start`, `split`, `y_price`, `y_spike_rel`, `has_occto`, `regime_2026`, LNG 미래 가격 관련 열은 입력하지 않는다.
- **추가 보수적 제외**: 당일 실측 날씨·당일 날씨 유래 변수 및 검증되지 않은 가격 이동평균. 예측 시점에 실제 이용 가능한 날씨 예보인지 확인되기 전에는 사용하지 않는다.

> ⚠️ 과거 1일·7일 lag 변수의 공표 시각은 별도 검증이 필요하다. 2025-11 이후 테스트의 실제 정답으로 모델을 고르지 않는다. 이 모델은 **11/18 모델 B 비교 전의 1차 임시 모델**이다.

## 1. 환경 확인
**VS Code**에서 이 노트북을 열고 우측 상단 Python 3.11 `.venv` 커널 선택. 패키지가 없다면 터미널에서 `python -m pip install -r requirements.txt` 실행.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from IPython.display import display

print('Python:', sys.version.split()[0], '| executable:', sys.executable)
print('LightGBM:', lgb.__version__, '| pandas:', pd.__version__)

## 2. 프로젝트 경로와 원본 CSV 탐색
현재 VS Code에 열린 최상위 폴더가 `project`라도 동작하도록 `second_project/11.17` 폴더를 탐색한다. 원본을 못 찾으면 `11.17/data/merged_30min.csv`로 복사하면 된다. 원본은 GitHub에 올리지 않는다.

In [ ]:
HERE = Path.cwd().resolve()
possible = [HERE, *HERE.parents]
project_candidates = []
for base in possible:
    project_candidates.extend([base / 'second_project' / '11.17', base])
PROJECT_DIR = next((p for p in project_candidates if p.name == '11.17' and (p / 'notebooks').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('second_project/11.17/notebooks 폴더를 찾지 못했습니다. VS Code에서 project 폴더를 열어 주세요.')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
FIGURES_DIR = PROJECT_DIR / 'figures'
for p in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 필요하다면 JEPX_DATA_PATH 환경변수로 CSV의 정확한 경로를 지정할 수 있음.
manual = os.environ.get('JEPX_DATA_PATH')
if manual:
    CSV_PATH = Path(manual).expanduser().resolve()
else:
    preferred = DATA_DIR / 'merged_30min.csv'
    if preferred.is_file():
        CSV_PATH = preferred
    else:
        matches = list(PROJECT_DIR.parent.rglob('merged_30min.csv'))
        if len(matches) != 1:
            raise FileNotFoundError(f'원본 CSV를 자동 선택할 수 없습니다. 발견: {matches}. {preferred}에 복사하거나 JEPX_DATA_PATH를 설정하세요.')
        CSV_PATH = matches[0]
print('프로젝트 폴더:', PROJECT_DIR)
print('사용 데이터:', CSV_PATH)

## 3. 데이터 읽기와 시간 순서 분리
기존 `split`에는 2025년 5~10월 검증 구간이 저장되어 있다. 혼동을 막기 위해 **`date`로 새로운 `split_updated`** 를 만들고 기존 `split`은 학습 변수로 사용하지 않는다.

In [ ]:
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
df['date'] = pd.to_datetime(df['date'], errors='raise')
required = {'date', 'slot', 'y_price', 'price_tokyo_lag1d'}
assert required.issubset(df.columns), f'필수 열 누락: {required-set(df.columns)}'
assert not df[['date', 'slot']].isna().any().any()
assert not df.duplicated(['date', 'slot']).any(), 'date+slot 중복이 있습니다.'
assert df['date'].between('2024-04-01', '2026-10-31').all()
assert df['slot'].between(1, 48).all()

df['split_updated'] = np.select(
    [df['date'] < '2025-09-01', df['date'] < '2025-11-01'],
    ['train','valid'],default='test'
)
train = df.loc[df['split_updated'] == 'train'].copy()
valid = df.loc[df['split_updated'] == 'valid'].copy()
test = df.loc[df['split_updated'] == 'test'].copy()
print('전체:', df.shape, '각 기간:', df['split_updated'].value_counts().to_dict())
print('기간:', {name:(sub.date.min().date(),sub.date.max().date()) for name,sub in [('train',train),('valid',valid),('test',test)]})
assert (len(train),len(valid),len(test)) == (24864, 2928, 17520), '예상 행 수가 다릅니다.'

## 4. 모델 입력 변수(X) 정하기
숫자를 예측하는 `y_price`는 정답(y)이지 입력(X)이 아니다. 아래는 **달력 7개 + 전날/7일 전 지연변수 42개 = 49개**만 사용한다. 당일 실측 날씨와 OCCTO는 제외한다.

In [ ]:
calendar_features = ['hour','weekday','month','is_holiday','is_offday','is_yearend','is_obon']
lag_features = [c for c in df.columns if c.endswith('_lag1d') or c.endswith('_lag7d')]
features = calendar_features + lag_features
forbidden = {'date','slot','slot_start','split','split_updated','y_price','y_spike_rel','has_occto','regime_2026','lng_yen_per_t_m1','lng_yen_per_t_m2'}
assert not forbidden.intersection(features)
assert not any(c.startswith('occto_') for c in features)
assert len(features) == 49, f'특징 개수가 다릅니다: {len(features)}'
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in features), '비숫자 입력 열이 있습니다.'
assert not df[features].isna().any().any(), '입력 변수에 결측이 있습니다.'
print('사용 변수:', len(features), '개')
print(features)

## 5. LightGBM 회귀 모델 A 학습
train만 학습하고 valid는 early stopping 및 평가에 사용한다. test는 학습이나 하이퍼파라미터 선택에 사용하지 않는다.

In [ ]:
model_params = dict(
    objective='regression', n_estimators=800, learning_rate=0.035,
    num_leaves=15, max_depth=-1,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.85, min_child_samples=35,
    reg_lambda=2.0, random_state=42, n_jobs=2,
    verbosity=-1, importance_type='gain'
)
model_A = lgb.LGBMRegressor(**model_params)
model_A.fit(
    train[features], train['y_price'],
    eval_set=[(valid[features], valid['y_price'])], eval_metric='l1',
    callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)]
)
print('학습 완료 / best_iteration:', model_A.best_iteration_)

## 6. 베이스라인 대비 검증 MAE 평가
`MAE = |정답 - 예측|의 평균`으로, 작을수록 좋다. 같은 valid 기간에서 두 모델을 비교한다.

In [ ]:
valid_pred_A = model_A.predict(valid[features])
baseline_mae = mean_absolute_error(valid['y_price'], valid['price_tokyo_lag1d'])
model_A_mae = mean_absolute_error(valid['y_price'], valid_pred_A)
improvement_pct = (baseline_mae - model_A_mae) / baseline_mae * 100
print(f'Baseline Valid MAE: {baseline_mae:.6f}')
print(f'Model A Valid MAE: {model_A_mae:.6f}')
print(f'MAE 변화율: {improvement_pct:+.2f}% 개선 (양수는 개선)')

metrics = pd.DataFrame([
    {'model':'Previous-day baseline','split':'valid','period':'2025-09-01~2025-10-31','n_rows':len(valid),'MAE':baseline_mae,'best_iteration':''},
    {'model':'LightGBM Model A','split':'valid','period':'2025-09-01~2025-10-31','n_rows':len(valid),'MAE':model_A_mae,'best_iteration':int(model_A.best_iteration_)}
])
metrics.to_csv(RESULTS_DIR / 'model_A_valid_metrics.csv', index=False, encoding='utf-8-sig')

valid_predictions = pd.DataFrame({
    'date':valid['date'].dt.strftime('%Y-%m-%d').to_numpy(),
    'slot':valid['slot'].to_numpy(),
    'y_price':valid['y_price'].to_numpy(),
    'baseline_price_tokyo_lag1d':valid['price_tokyo_lag1d'].to_numpy(),
    'y_price_pred':valid_pred_A
})
valid_predictions['baseline_abs_error'] = (valid_predictions['y_price']-valid_predictions['baseline_price_tokyo_lag1d']).abs()
valid_predictions['model_A_abs_error'] = (valid_predictions['y_price']-valid_predictions['y_price_pred']).abs()
valid_predictions.to_csv(RESULTS_DIR/'model_A_valid_predictions_PRIVATE.csv',index=False,encoding='utf-8-sig')
display(metrics)

## 7. 검증 결과 시각화
기준선과 모델 A의 동일한 검증 구간 성능, 오차 시간대, 실제값과 예측값, 주요 변수 15개를 확인한다. 변수 중요도는 인과관계를 증명하지 않는다.

In [ ]:
from matplotlib import font_manager
# 한국어 글꼴이 설치된 경우 사용. 없으면 영문 차트 제목으로 대체 가능.
font_names = {f.name for f in font_manager.fontManager.ttflist}
if 'NanumSquare' in font_names:
    plt.rcParams['font.family'] = 'NanumSquare'
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(8,4.8))
bs = ax.bar(['Baseline','LightGBM Model A'],[baseline_mae, model_A_mae])
for bar, value in zip(bs,[baseline_mae,model_A_mae]):
    ax.text(bar.get_x()+bar.get_width()/2, value+0.02, f'{value:.4f}',ha='center')
ax.set_ylim(0,max(baseline_mae,model_A_mae)*1.2)
ax.set_title('Validation MAE: Baseline vs Model A');ax.set_ylabel('MAE (lower is better)')
fig.tight_layout();fig.savefig(FIGURES_DIR/'박경필_회귀_모델A_vs_베이스라인.png',dpi=160,bbox_inches='tight');plt.show()

hourly = valid_predictions.assign(hour=(valid_predictions['slot']-1)//2).groupby('hour')[['baseline_abs_error','model_A_abs_error']].mean()
fig,ax=plt.subplots(figsize=(10,4.8))
ax.plot(hourly.index,hourly['baseline_abs_error'],'o-',label='Baseline')
ax.plot(hourly.index,hourly['model_A_abs_error'],'o-',label='LightGBM Model A')
ax.set_xticks(range(24));ax.set_xlabel('Hour (0-23)');ax.set_ylabel('MAE');ax.set_title('Validation hourly MAE');ax.legend();ax.grid(alpha=.25)
fig.tight_layout();fig.savefig(FIGURES_DIR/'박경필_회귀_모델A_시간대별MAE.png',dpi=160,bbox_inches='tight');plt.show()

fig,ax=plt.subplots(figsize=(7,6))
ax.scatter(valid['y_price'], valid_pred_A, s=12,alpha=.23)
mn = float(min(valid['y_price'].min(),valid_pred_A.min()))
mx = float(max(valid['y_price'].max(),valid_pred_A.max()))
ax.plot([mn,mx],[mn,mx],'r--',label='Perfect prediction')
ax.set_xlabel('Actual price');ax.set_ylabel('Predicted price');ax.set_title('Validation: Actual vs Model A');ax.grid(alpha=.2);ax.legend()
fig.tight_layout();fig.savefig(FIGURES_DIR/'박경필_회귀_모델A_실제값예측값.png',dpi=160,bbox_inches='tight');plt.show()

feature_importance = pd.DataFrame({'feature':features,'gain':model_A.feature_importances_}).sort_values('gain',ascending=False)
feature_importance.to_csv(RESULTS_DIR/'model_A_feature_importance.csv', index=False,encoding='utf-8-sig')
top15 = feature_importance.head(15).iloc[::-1]
fig,ax=plt.subplots(figsize=(10,6))
ax.barh(top15['feature'],top15['gain']);ax.set_xlabel('Gain importance');ax.set_title('LightGBM Model A feature importance TOP 15')
fig.tight_layout();fig.savefig(FIGURES_DIR/'박경필_회귀_모델A_중요변수_TOP15.png',dpi=160,bbox_inches='tight');plt.show()

## 8. 테스트 기간 **예측값만** 생성 (임시)
2025-11~2026-10의 결과는 Model B 비교 전까지 **PROVISIONAL/임시**로 표시한다. 이 단계에서 테스트의 `y_price`와 예측값을 비교하거나 튜닝하지 않는다.

In [ ]:
test_pred_A = model_A.predict(test[features])
test_prediction_df = pd.DataFrame({
    'date':test['date'].dt.strftime('%Y-%m-%d').to_numpy(),
    'slot':test['slot'].to_numpy(),
    'y_price_pred':test_pred_A
})
assert len(test_prediction_df) == 17520
assert test_prediction_df[['date','slot']].duplicated().sum() == 0
assert test_prediction_df['y_price_pred'].notna().all()
test_prediction_path = RESULTS_DIR/'model_A_test_predictions_PROVISIONAL_PRIVATE.csv'
test_prediction_df.to_csv(test_prediction_path,index=False,encoding='utf-8-sig')
print('테스트 예측값 저장:', test_prediction_path, '| 행 수:', len(test_prediction_df))

## 9. 팀원 분류 CSV와 `date + slot` 기준 병합 (선택)
`신현지_분류_테스트예측.csv`를 프로젝트의 `data/`에 놓고 실행한다. `y_price_pred` 열이 모두 빈 경우에만 채운다. 분류 확률 등 나머지 열은 보존한다.

> 통합 CSV는 **모델 B 비교 전 임시 산출물**이다. 원본 팀원 파일을 덮어쓰지 않는다.

In [ ]:
team_path = DATA_DIR/'신현지_분류_테스트예측.csv'
if not team_path.exists():
    print('팀원 CSV가 없어 결합은 건너뜁니다. 나중에', team_path, '에 파일을 복사한 뒤 이 셀만 다시 실행하세요.')
else:
    teammate = pd.read_csv(team_path,encoding='utf-8-sig')
    original_columns = teammate.columns.tolist()
    assert {'date','slot','y_price_pred'}.issubset(teammate.columns)
    assert teammate['y_price_pred'].isna().all(), '기존 y_price_pred에 값이 있으므로 덮어쓰지 않습니다.'
    assert not teammate[['date','slot']].isna().any().any()
    assert teammate[['date','slot']].duplicated().sum() == 0
    teammate['date'] = pd.to_datetime(teammate['date'],errors='raise').dt.strftime('%Y-%m-%d')
    joined = teammate.drop(columns=['y_price_pred']).merge(
        test_prediction_df, on=['date','slot'], how='left', validate='one_to_one', indicator=True
    )
    assert joined['_merge'].eq('both').all() and len(joined)==len(teammate)
    joined = joined.drop(columns=['_merge'])[original_columns]
    assert joined['y_price_pred'].notna().all()
    out=RESULTS_DIR/'신현지_분류_테스트예측_회귀A_임시_PRIVATE.csv'
    joined.to_csv(out,index=False,encoding='utf-8-sig')
    print('임시 통합 결과:',len(joined),'행 /',len(joined.columns),'열 / 위치:',out)

## 10. 분석 기록 및 다음 작업
- 이 Notebook은 회귀 **모델 A 1차** 학습·검증 결과다. 성능과 예측값은 실행 환경/라이브러리 버전에 따라 미세하게 다를 수 있다.
- **11/18 예정**: 모델 B(OCCTO 포함) 학습, 모델 A/B 공통 검증 기간으로 비교, 최종 모델 확정.
- 테스트 데이터의 정답으로 튜닝하지 않는다. 최종 모델 확정 후 예측값을 확정한다.
- GitHub 공개 전 원본 CSV·개별 가격 데이터·모델 출력·팀원의 파일 공개 권한을 확인한다.

**공식 문서**: [LightGBM LGBMRegressor](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html), [scikit-learn 데이터 누출](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), [pandas merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).